# Homework - 2

In this homework, you will learn:

1. How to code the [Transformer](https://arxiv.org/abs/1706.03762) architecture, an essential skill of the modern DL engineer.
2. How to work with different [HuggingFace](https://huggingface.co/) tools.
3. How to use LLM APIs.
4. Some techniques for efficient LLM fine-tuning.

As in the previous homework, we expect you to create configurable code and put all the cells in the chronological order, so your solution can be run sequentially. Also, you must use [Comet](https://www.comet.com/site/) for tracking (both for HuggingFace and PyTorch training tasks) and provide a Comet Report together with the `ipynb` notebook in your submission. 

LLM-generated code/analysis for tasks is strongly discouraged.

Penalties will be applied if these rules are not followed.

**Note** This homework will require you to use many tokens (HF, Groq, etc.). Do not submit them with your notebook. It is private information. Leakage of tokens will be penalized.

**Hint** Create a `.py` or `.env` file with all your tokens. Upload it to your working space (Kaggle/Colab) and import tokens from the file without ever exposing their values: `from private_tokens import HF_TOKEN`, for example.

## Part I: Machine Translation

In this task, your goal is to learn how to code Transformer architecture and do autoregressive (AR) generation. We will focus on **Ru$\to$En** text translation.

### (Task 1) (1 pts)

Let's start by looking at the data and process it. Download the [En $\to$ Ru subset of Opus Books](https://huggingface.co/datasets/Helsinki-NLP/opus_books) and sample some examples from it to see how it looks.

In [ ]:
# YOUR CODE HERE

Split the dataset into a 9:1 train/test partitions. Use `seed=1`.

In [ ]:
# YOUR CODE HERE

For training a machine translator, we need to convert raw strings into token ID lists. So, we need a `tokenizer`. 

Let's learn how to train such a `tokenizer` from scratch. First, have a look at [this tutorial](https://huggingface.co/learn/llm-course/chapter6/8). Let's do the **BPE** variant with some differences:

1. Pre-tokenize using WhiteSpace.
2. Normalize all unicode characters (NFKC).
3. Add all the required special tokens needed for Machine Translation (recall that your model needs to know when the sentence starts and ends).
4. Use `WordPiece` decoder to correct decoding punctuation (try without it and see the issue)

For the vocabulary size, set a value between 8-16k.

Write down all the required code and train your tokenizer on all Ru/En text in the **train** partition of your dataset. Do not forget to convert the tokenizer into `FastTokenizer` using `PreTrainedTokenizerFast`.

In [ ]:
# YOUR CODE HERE

Sample some examples from the dataset and encode / decode them. Verify that your code is working.

In [ ]:
# YOUR CODE HERE

### (Task 2) (1 pts)


Create your own implementation of the Transformer architecture. Use sinusoidal positional encoding and an **encoder-decoder** style. You **are not allowed to use `nn.Transformer` and other `PyTorch` Transformer-related layers; other lirbraries, e.g. `HuggingFace`, are also banned**. You can only use `nn.Linear`, activation layers, and `nn.LayerNorm` with `nn.Dropout`. Use pre-norm style

$$
y_t = x_t + \text{MHA}(\text{LN}(x_t)), \qquad x_{t+1} = y_t + \text{FFN}(\text{LN}(y_t))
$$

Every Deep Learning Engineer should at least once try to write `Transformer` on their own, so do not try to cheat.

In [ ]:
# YOUR CODE HERE

### (Task 3) (0.5 pts)

Create a `translate` function that will take English text as input and translate in using your model in a **greedy-decoding** way, i.e., at each prediction step you should take the token with the highest probability.

In [ ]:
# YOUR CODE HERE

### (Task 4) (1.0 pts)

Create a `Dataset` class to train Transformer on it. _We want to translate Russian text into English (**Ru $\to$ En**)_.

Train your `Transformer` on the train partition you obtained in Task 1 and achieve at least $7.5$ BLEU score (use `sacrebleu` lib) on the test split. For the training loop, write your own code, do not use ready-to-use Trainers. Do not forget about logging, `collate_fn` for padding, and corresponding masking.

**Hint** The `CrossEntropyLoss` has an `ignore_index` field. If you set it to `pad_token_id`, you won't need to handle padding in the loss yourself. 

**Hint 2** It might be useful to log BLEU for train partition (or subset) as well.


In [ ]:
# YOUR CODE HERE

Run the final evaluation and show the BLEU score:

In [ ]:
# YOUR CODE HERE

### (Task 5) (0.75 pts)

In the LLM lecture, we discussed that most tasks can be re-formulated in the decoder-only style. Create a prefix-decoder version of your `Transformer` and train it on the same data. 

**Note** you need to separate Russian text from the English version. One can do it using a `SEP` special token in the tokenizer. But we will investigate another option - continuous embedding. When provided to the Transformer, all tokens are converted to embeddings based on token ids, but you do not need to have an id to get an embedding. Indeed, you can just create a parameter in your transformer and concatenate it with the input Russian sequence. This trainable parameter will tell the model that subsequent tokens should be in English.

$$
[h_{\text{Ru}}, h_{\text{trainable sep. embedding}}, h_{\text{En}}]
$$

**Hint** A simple way to calculate loss only on English tokens in the prefix decoder is to utilize the `ignore_index` field of `CrossEntropyLoss`. Think what should you do with the target you pass to the loss to avoid calculating loss on Russian tokens.


**For fair comparison, make sure to use the same training setup as the encoder-decoder variant and use similar number of parameters.**

In [ ]:
# YOUR CODE HERE

### (Task 6) (0.75 pts)

Model analysis can be done in two ways. "Quantitative" is based on metrics (scores) and is usually conducted over the whole dataset to get some objective measure. "Qualitative" is more subjective but required to see if the scores make sense. The latter evaluation means showing some examples of model generations and manually analyzing their quality. 

**Quantitative** What other metrics apart from BLEU can be used to evaluate the performance of machine translation? Choose at least 2 other metrics and calculate their scores for both Transformer models.

**Qualitative** Sample 5 translations for each model. 



In [ ]:
# YOUR CODE HERE

Explain the choice of your metrics and how they differ from each other. Do they allow to evaluate different aspects of translation? Which is primary (the most important)? Do metrics reflect what you see in the generated samples? What can you say about the quality of samples for each model, how do they differ? Then, compare encoder-decoder and prefix-decoder variants in terms of convergence speed, model quality, etc.. Explain the observations and write down a conclusion.

**YOUR RESPONSE HERE**

## Part II: Detection of Machine-generated Text

In the LLM era, synthesized texts appear everywhere. Your goal is to design and train a text deepfake detector. 

Deepfake detection (in any modality) posses significant challenges related to out-of-domain generatlization, i.e., having high prediction accuracy on unseen data likely generated via unseen LLMs. Indeed, new synthesizers are developed every week and it is impossible to have samples from all of them in the training set, especially for proprietary models. Furthermore, collecting enough examples for each LLM is time-consuming and expensive.

This puts the task in the classic setup of scarce amount of labeled data with the need to still having robust performance in the wild. As we saw in class, a common approach would be to use a pre-trained large backbone that already knows how the real data should look like and how to extract meaningful concepts and, then, fine-tune this backbone on our small labeled dataset. 

In this homework, we will do exactly this. We will take a pre-trained [RoBERTa](https://huggingface.co/FacebookAI/roberta-base) and fine-tune it on a self-collected dataset of human\AI text pairs.

For the dataset, the **train partition will be collected by yourself**. You will evaluate your model on the **validation** split provided [here](https://huggingface.co/datasets/Blinorot/DLR_HW2_VAL) (label = 1 means human). All tasks below must log accuracy on this validation split. Then, in the bonus task, you will have a comptetition on a [test set](https://huggingface.co/datasets/Blinorot/DLR_HW2_TEST). **You cannot use provided datasets for training**.

In [ ]:
# download the validation and test splits

### (Task 7) Dataset Collection [2 pts]

We need to have both real text examples and AI-generated examples, so the model learns to differentiate between them. For the real partition, find any dataset (**you can use train partition only**) on HuggingFace that contains **only human-written** text. Make sure you use only **training partitions** of the datasets to avoid contamination.

For AI-generated texts, all of them **must be collected yourself** using`openai` + [vllm](https://github.com/vllm-project/vllm) serving with HuggingFace models (any that will fit in your Kaggle/Colab GPU) or **free** [Groq](https://groq.com/) APIs (see the documentation to understand how to use `groq`).

Combine all your data (real + generated) into a HuggingFace `datasets.DatasetDict` and upload it on you HuggingFace account. Attach the collected dataset link in the homework submission system.

**Hint** investigate the validation split we provided to understand what type of texts you will need to handle. Provided texts are generated by many LLMs and techniques.

Describe your dataset creation strategy. What libraries and models do you use, what real data do you use, what prompts do you use, etc. Make sure to include your reasoning behind each design choice.

**YOUR STRATEGY DESCRIPTION HERE:** 

In [ ]:
# YOUR CODE FOR CREATING AND SAVING THE DATASET

Provide the final statistics for your dataset: how many examples per each AI tool do you have, how many examples total, what data domains do you cover (if multiple), etc.

In [ ]:
# CALCULATE STATISTICS

**YOUR DATASET STATISTICS DESCRIPTION HERE**

### (Task 8) RoBERTa fine-tuning [1 pts]

Using your collected dataset and `transformers` library, download the `RoBERTa` [weights](https://huggingface.co/FacebookAI/roberta-base) and fine-tune it on the collected dataset with the `Trainer` class. 

Take a subset of your dataset and use it for validation. Log scores both on your's and teacher's validation. Clearly separate these two metrics in your logs.

To get full points, we expect you to achieve at least 70% accuracy on the [teacher's validation split](https://huggingface.co/datasets/Blinorot/DLR_HW2_VAL). **Hint:** your accuracy will also depend on the quality of your collected dataset, so you might want to improve it.

In [ ]:
#YOUR CODE HERE

Run the final evaluation on [teacher's validation](https://huggingface.co/datasets/Blinorot/DLR_HW2_VAL) and print the score.

In [ ]:
# YOUR CODE HERE

### (Task 9) LoRA fine-tuning [2 pts]

Fine-tuning the full model, especially when it is large, is expensive. In particular, for LLMs, it is common that you cannot even fit the model into your GPU if you try to fine-tune it fully. This is why [performance-efficient fine-tuning (PEFT)](https://arxiv.org/abs/2403.14608) techniques have been developed over the years.

[Low-Rank Adaptation (LoRA)](https://arxiv.org/abs/2106.09685) is one of such techniques. The idea behind it is rather simple. Given a weight matrix $W_0 \in \mathbb{R}^{d\times k}$ in a `nn.Linear` layer, a conventional fine-tuning will update it by adding some shift $\Delta W \in \mathbb{R}^{d\times k}$ as follows: $W = W_0 + \Delta W$.

LoRA says that instead of directly calculating $\Delta W$ one can freeze $W_0$ and parametrize the update as $$\Delta W = AB,$$ where $A \in \mathbb{R}^{d\times r}, B \in \mathbb{R}^{r\times k}$ are low-rank matrices with rank $r << \min(d,k)$. 

Now, the forward process becomes $$f(x) = W_0x + ABx$$ instead of just $W_0x$. 

To ensure that we do not shift from the original model at the beginning of training, we initialize $A$ with a standard random initialization (e.g., Gaussian) and $B$ with a full-zero matrix, so $AB=0$ at start. Since the $r$ is small, these $A, B$ require calculating gradients for much smaller number of parameters (compared to $W$) and can be computed faster leading to reduced memory consumption and increased speed. Besides, during inference, one can merge $A, B$ inside $W_0$ simply defining $W_0 := W_0 + AB$, so the model performs inference exactly like a standard linear layer with no additional computational overhead

The interesting finding from the paper is that low-rank update is commonly enough, since pretrained models already contain most of the required knowledge, and downstream tasks only require small adjustments within a low-dimensional subspace.


In this task, we ask you to implement LoRA **yourself from scratch**. Using implementations from the packages/web is **prohibited**.

Create a `PyTorch` LoRA wrapper class which you will apply on a linear layer. Find Query,Key,Value projection layers in the `RoBERTa` model and wrap them with your class ($r=8$). Then, freeze all `RoBERTa` weights and fine-tune the model with only LoRA weights being updated.

**Hint:** to freeze the model in `PyTorch`, you can do:

```python
for param in model.parameters():
    param.requires_grad = False
```

This will freeze all the parameters in your model. Make sure to unfreeze LoRA weights by setting `requires_grad` to `True`.

**Hint** to find names of the attention parameters, print the model 

In [ ]:
# YOUR CODE HERE

Run the final evaluation on [teacher's validation](https://huggingface.co/datasets/Blinorot/DLR_HW2_VAL) and print the score. We expect you to achieve at least $70\%$ accuracy on the validation split

In [ ]:
# YOUR CODE HERE

Another important factor is speed and memory consumption. LoRA should reduce the computational burden. Let's validate it. Create a simple function `train_one_step` which will:

1. Run the model
2. Calculate loss
3. Calculate gradients and update the model with an optimizer

Then measure the time needed for this step using `time` library and the memory needed using `torch.cuda.reset_peak_memory_stats()`, `torch.cuda.memory_allocated()`, and `torch.cuda.max_memory_allocated()`. Do this for the setup where the whole RoBERTa is updated and for the setup where only LoRA weights are updated.


**Hint:** use `torch.cuda.synchronize()` before and after calling your function to ensure that all previous operations have stopped before your run and your own operation will be finished before the function ends. Also note that GPUs have a warm-up stage: the first few operations are warming up the GPU and might take longer than subsequent operations. So, you have to run your function several times and skip the first 2-3 times.

In [ ]:
# YOUR CODE HERE

Analyze the time/memory measurements and the accuracy/loss curves (from LoRA and full fine-tuning). Compare performance on your validation and on teacher's validation. Analyze and explain the differences. Make the conclusions about the effectiveness of LoRA in terms of model performance and training costs.

**YOUR RESPONSE HERE**

### (Bonus-1) (up to 1.5 pts) 

Try different techniques to improve the quality of your model. This might be creating a better or larger dataset or something less straightforward. If your approach involves some sort of training (fine-tuning or extraction of embeddings, e.g.), you can only use RoBERTa checkpoint we had before. Adding extra modules on top of RoBERTa is allowed if new modules are trained from scratch. **You cannot use any model that was trained\tuned to do text deepfake detection from the web and you cannot fine-tune any other architecture yourself**. But, if no training is involved and the model was not trained to do deepfake detection, you may use it. Combining (ensembling) different solutions of your authorship is allowed. You can use any amount of real data (**only from the training partitions of the datasets**) but all AI data must be generated yourself.

To get bonus points you must provide a Comet report explaining (attach in the homework submission system):

1. What experiments have you tried and the motivation behind each experiment
2. Loss/accuracy curves for each of the experiments
3. Your analysis of plots and the experiments you did. Make sure to justify the experiment design and why it gave/did not give improvement.

Make sure that your report is well-structured, the curve labels are clear and meaningful, etc. 

The bonus grade will depend on the quality of the report and how creative your experiments are.

**Hint** you may find useful concepts to try by reading machine-generated text detection literature. To find it, use corresponding keywords in the [Google Scholar](https://scholar.google.com/). Start by looking at surveys. Survey is a comprehensive review of a large number of articles presented in the grouped and structured way allowing the reader to quickly get an overview of the field, understand the basic concepts and techniques and how researchers usually tackle the problem. You may also find inspiration in adjacent research fields, e.g., anomaly detection, audio/video/code deepfake detection, domain adaptation, etc.

In [ ]:
#YOUR CODE FOR BONUS HERE

Run your best solution and show the score on teacher's validation.

In [ ]:
# YOUR CODE HERE

### (Bonus-2) (up to 1 pts)

We provide you with a [test set (without labels)](https://huggingface.co/datasets/Blinorot/DLR_HW2_TEST). Evaluate your model on it and submit predictions to the Kaggle Competition (link is provided in the course channel). 

The first 2 places will get $1$ extra point. The 3rd and 4th -- $0.75$. The 5-th and 6-th -- $0.5$.

Provide your nickname in the homework submission system. Your nickname must start with your surname (here, nickname means the name of your team, **with only 1 member -- yourself**, in the competition). (If you indicate a nickname of another person, you will be banned)

In [ ]:
# YOUR TEST EVAL HERE

Also, provide results of the same solution on the teacher's validation split below

In [ ]:
# YOUR CODE HERE